# Missing data and class imbalance: worked solution

Worked solution for the exercise in the lesson **[Missing data and class imbalance](https://ramkumarmd.com/medical-ai/machine-learning/missing-data-and-imbalance)** (*Machine learning* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](06-missing-data-and-imbalance.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** In clinical data, whether a value is missing often depends on how ill the patient seemed, so dropping incomplete records or ignoring missingness can bias a model. For prediction, imputation inside the modelling pipeline, sometimes with a 'was it measured?' flag, usually works best, as long as testing practices stay stable. Rare outcomes rarely need 'correcting': oversampling or reweighting leaves AUROC unchanged and makes predicted risks far too high.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q imbalanced-learn

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Informative missingness, in code

Synthetic patients in whom lactate is measured mainly when severity is high.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(13)
n = 8000
severity = rng.normal(0, 1, n)
lactate = (1.5 + 0.8 * severity + rng.normal(0, 0.5, n)).clip(0.3).round(1)
died = rng.binomial(1, 1 / (1 + np.exp(-(-3.0 + 1.6 * severity))))
measured = rng.binomial(1, 1 / (1 + np.exp(-(-0.5 + 1.8 * severity)))).astype(bool)
df = pd.DataFrame({"age": rng.integers(18, 90, n),
                   "heart_rate": (85 + 6 * severity + rng.normal(0, 12, n)).round(),
                   "lactate": np.where(measured, lactate, np.nan),
                   "died": died})
print(f"Lactate missing in {df.lactate.isna().mean():.0%} of patients")
print(df.groupby(df.lactate.isna()).died.mean().rename("death rate").round(3))

# Lactate missing in 58% of patients
# lactate
# False    0.184
# True     0.042
# Name: death rate, dtype: float64

### Dropping, imputing and flagging

Continuing from the previous block: three ways to handle missing lactate, evaluated on the same test set.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = df[["age", "heart_rate", "lactate"]], df.died
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=0, stratify=y)

def evaluate(columns, imputer, label):
    model = make_pipeline(imputer, StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_train[columns], y_train)
    p = model.predict_proba(X_test[columns])[:, 1]
    print(f"{label:<34} AUROC {roc_auc_score(y_test, p):.3f}")

evaluate(["age", "heart_rate"], SimpleImputer(), "Lactate dropped")
evaluate(["age", "heart_rate", "lactate"], SimpleImputer(strategy="median"),
         "Median imputation")
evaluate(["age", "heart_rate", "lactate"],
         SimpleImputer(strategy="median", add_indicator=True),
         "Median imputation + missing flag")

# Lactate dropped                    AUROC 0.658
# Median imputation                  AUROC 0.768
# Median imputation + missing flag   AUROC 0.784

### Class imbalance: usually not a problem to fix

A rare outcome (about 5%) with and without class reweighting.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(4)
n = 20000
X = rng.normal(size=(n, 4))
y = rng.binomial(1, 1 / (1 + np.exp(-(-3.5 + X @ np.array([0.9, 0.6, 0.4, 0.0])))))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=0, stratify=y)
print(f"Outcome rate {y.mean():.1%}")

for label, weights in [("No correction", None), ("class_weight='balanced'", "balanced")]:
    model = LogisticRegression(class_weight=weights).fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    print(f"{label:<24} AUROC {roc_auc_score(y_test, p):.3f}   "
          f"mean predicted risk {p.mean():.1%}   observed {y_test.mean():.1%}")

# Outcome rate 4.7%
# No correction            AUROC 0.811   mean predicted risk 4.7%   observed 4.7%
# class_weight='balanced'  AUROC 0.811   mean predicted risk 37.4%   observed 4.7%

## Exercise

**Goal:** Handle informative missingness and a rare outcome properly, and measure the consequences of common shortcuts.

**Data:** Synthetic patients generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the lactate example and add a complete-case model; compare its mean predicted risk with the observed rate in the full test set.

In [ ]:
# The imbalance example above reused the names X, y, X_train...; rebuild the lactate split first.
from sklearn.metrics import brier_score_loss

Xl, yl = df[["age", "heart_rate", "lactate"]], df.died
Xl_train, Xl_test, yl_train, yl_test = train_test_split(Xl, yl, test_size=0.3, random_state=0, stratify=yl)

def fit_eval(imputer, label, X_tr=Xl_train, y_tr=yl_train, X_te=Xl_test, y_te=yl_test):
    model = make_pipeline(imputer, StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
    p = model.predict_proba(X_te)[:, 1]
    print(f"{label:<36} AUROC {roc_auc_score(y_te, p):.3f}  Brier {brier_score_loss(y_te, p):.4f}  "
          f"mean predicted {p.mean():.1%} (observed {y_te.mean():.1%})")
    return model

flagged = fit_eval(SimpleImputer(strategy="median", add_indicator=True), "Median imputation + missing flag")

# Complete-case analysis: keep only patients whose lactate was measured
cc = Xl_train.lactate.notna()
cc_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xl_train[cc], yl_train[cc])
measured_test = Xl_test.lactate.notna()
p_cc = cc_model.predict_proba(Xl_test[measured_test])[:, 1]
print(f"\nComplete-case model: trained on {cc.sum()} of {len(cc)} patients; "
      f"mean predicted {p_cc.mean():.1%} vs observed {yl_test.mean():.1%} in the full test set")

The complete-case model learns from the sicker patients who had lactate measured, so its average
prediction matches *their* death rate, nearly double the rate in the whole population. It also cannot
produce a prediction at all for the majority whose lactate is missing.

### Step 2

Replace median imputation with IterativeImputer and compare AUROC and calibration.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (needed to import IterativeImputer)
from sklearn.impute import IterativeImputer

fit_eval(SimpleImputer(strategy="median"), "Median imputation")
fit_eval(IterativeImputer(random_state=0), "Iterative imputation")
_ = fit_eval(IterativeImputer(random_state=0, add_indicator=True), "Iterative imputation + missing flag")

Iterative imputation predicts lactate from age and heart rate. That helps a little, but it assumes the
values are missing at random given those variables, which is false here: lactate is missing *because*
patients look well. The missingness flag captures that directly, and it is what improves the model.

### Step 3

Simulate a change in practice in which lactate is measured in everyone, and evaluate the model with the missingness flag on this new data.

In [ ]:
# New practice: lactate is measured in everyone. Same patients' biology, different ordering behaviour.
rng = np.random.default_rng(14)
m = 4000
sev = rng.normal(0, 1, m)
new = pd.DataFrame({
    "age": rng.integers(18, 90, m),
    "heart_rate": (85 + 6 * sev + rng.normal(0, 12, m)).round(),
    "lactate": (1.5 + 0.8 * sev + rng.normal(0, 0.5, m)).clip(0.3).round(1),   # never missing now
    "died": rng.binomial(1, 1 / (1 + np.exp(-(-3.0 + 1.6 * sev)))),
})
p_new = flagged.predict_proba(new[["age", "heart_rate", "lactate"]])[:, 1]
print(f"Flag model after the practice change: AUROC {roc_auc_score(new.died, p_new):.3f}, "
      f"mean predicted {p_new.mean():.1%} vs observed {new.died.mean():.1%}")

The model learned "a missing lactate means low risk". When everyone is tested, nobody gets that reassurance,
so the model now over-predicts (about 12.6% predicted against 11.0% observed here) even though the patients
themselves have not changed. AUROC rises, because a real lactate value is now available for everyone, so a
discrimination check alone would not catch the problem. A model that relies on *ordering behaviour* must be
monitored for calibration, and refitted, when that behaviour changes.

### Step 4

Reproduce the imbalance example, then try random oversampling and SMOTE (imbalanced-learn); compare AUROC, calibration plots and Brier scores.

In [ ]:
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.calibration import CalibrationDisplay

# X, y, X_train... here are the imbalance example's (4.7% outcome rate)
samplers = {"No correction": None,
            "Random oversampling": RandomOverSampler(random_state=0),
            "SMOTE": SMOTE(random_state=0)}
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, sampler in samplers.items():
    Xs, ys = (X_train, y_train) if sampler is None else sampler.fit_resample(X_train, y_train)
    p = LogisticRegression().fit(Xs, ys).predict_proba(X_test)[:, 1]
    print(f"{name:<20} AUROC {roc_auc_score(y_test, p):.3f}  Brier {brier_score_loss(y_test, p):.4f}  "
          f"mean predicted {p.mean():.1%}")
    CalibrationDisplay.from_predictions(y_test, p, n_bins=10, strategy="quantile", ax=ax, name=name)
plt.show()

Resampling leaves AUROC essentially unchanged (the ranking of patients is the same) but makes the
predicted risks far too high, because the model is trained as if half of all patients had the outcome.
The Brier score gets worse accordingly. If you need a different trade-off between missed cases and false
alarms, change the decision threshold instead (next step).

### Step 5

For the uncorrected model, choose a threshold that gives 80% sensitivity and report the resulting PPV.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

p = LogisticRegression().fit(X_train, y_train).predict_proba(X_test)[:, 1]
fpr, tpr, thr = roc_curve(y_test, p)
t80 = thr[np.argmax(tpr >= 0.80)]
tn, fp, fn, tp = confusion_matrix(y_test, p >= t80).ravel()
print(f"Threshold {t80:.3f}: sensitivity {tp / (tp + fn):.2f}, specificity {tn / (tn + fp):.2f}, "
      f"PPV {tp / (tp + fp):.2f}, flagged {(p >= t80).mean():.0%} of patients")

To find 80% of the patients with this 4.7%-prevalence outcome, the model must flag a large share of
everyone, and most flagged patients will not have it. That trade-off, not the imbalance itself, is what
clinicians need to see when deciding whether a model is useful.

---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/missing-data-and-imbalance](https://ramkumarmd.com/medical-ai/machine-learning/missing-data-and-imbalance). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.